<a href="https://colab.research.google.com/github/beneducarrco99/Laboratorio_2_Semana_2_Ciencia_de_Datos_Carrasco_Benjamin/blob/main/S2_Notebook_Laboratorio_CDD_Carrasco_Benjamin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 2 — Calidad y preparación de tu conjunto de datos
**Ciencia de Datos · Semana 2 · Advance UCSC**

| | |
|---|---|
| **Resultado de aprendizaje** | RA1 (cierre) · RA3 (transversal) |
| **Duración estimada** | 1 h asincrónica + 1 h de trabajo autónomo |
| **Insumo** | El conjunto de datos que elegiste en la semana 1 |
| **Producto** | Conjunto limpio + bitácora completada + avance versionado |
| **Estudiante** | _Benjamín Carrasco Concha_ |

---

Este laboratorio **no trae los datos**: los traes tú. Todo lo que hagas aquí es el cuerpo de la **Entrega 1**, que rindes al cierre de esta semana.

Ten abierta en paralelo la planilla `S2_Bitacora_Decisiones_CDD.xlsx`. Escríbela **mientras trabajas**, no al final: reconstruir después por qué tomaste una decisión es mucho más difícil de lo que parece.

In [1]:
import pandas as pd
import numpy as np

RUTA = ''
SEMILLA = 20262
np.random.seed(SEMILLA)

ARCHIVO = '2025-sociedades-por-fecha-rut-constitucion.xlsx'   # <- reemplaza
SEPARADOR = ''                        # <- ajusta si corresponde

df = pd.read_excel(ARCHIVO)
print('Forma:', df.shape)
df.head()

Forma: (202406, 15)


,ID,RUT,Razon Social,Fecha de actuacion (1era firma),Fecha de registro (ultima firma),Fecha de aprobacion x SII,Anio,Mes,Comuna Tributaria,Region Tributaria,Codigo de sociedad,Tipo de actuacion,Capital,Comuna Social,Region Social
0,5597762,78081182-K,Directorio Pesticidas.cl SpA,2025-01-01,2025-01-01,2025-01-01,2025,Enero,MACUL,13,SpA,CONSTITUCIÓN,10000,MACUL,13.0
1,5597811,78081198-6,Apicola Navarro SpA,2025-01-01,2025-01-01,2025-01-01,2025,Enero,CHEPICA,6,SpA,CONSTITUCIÓN,1500000,CHEPICA,6.0
2,5598152,78081247-8,Corporación Pedrazzini SpA,2025-01-01,2025-01-01,2025-01-01,2025,Enero,MACUL,13,SpA,CONSTITUCIÓN,1000000,MACUL,13.0
3,5590234,78081257-5,Artesania Autoctona Angel SpA,2025-01-01,2025-01-01,2025-01-01,2025,Enero,ANTOFAGASTA,2,SpA,CONSTITUCIÓN,3000000,ANTOFAGASTA,2.0
4,5584967,78081269-9,Constructora Piedra Preciosa SpA,2025-01-01,2025-01-01,2025-01-01,2025,Enero,ANTOFAGASTA,2,SpA,CONSTITUCIÓN,5000000,ANTOFAGASTA,2.0


In [2]:
df.dtypes

,0
ID,int64
RUT,object
Razon Social,object
Fecha de actuacion (1era firma),datetime64[ns]
Fecha de registro (ultima firma),datetime64[ns]
Fecha de aprobacion x SII,datetime64[ns]
Anio,int64
Mes,object
Comuna Tributaria,object
Region Tributaria,int64


## Parte 1 · Diagnóstico de las cinco dimensiones

Antes de limpiar nada, hay que saber qué tienes. Completa la hoja **«Resumen de calidad»** de la bitácora con los resultados de esta parte.

### 1.1 Completitud

In [3]:
faltantes = df.isna().sum().to_frame('faltantes')
faltantes['pct'] = (faltantes['faltantes'] / len(df) * 100).round(2)
faltantes.sort_values('faltantes', ascending=False)

,faltantes,pct
Comuna Social,3,0.0
Region Social,3,0.0
ID,0,0.0
Fecha de actuacion (1era firma),0,0.0
Fecha de registro (ultima firma),0,0.0
RUT,0,0.0
Razon Social,0,0.0
Anio,0,0.0
Fecha de aprobacion x SII,0,0.0
Mes,0,0.0


### ✏️ Actividad 1.1

Elige la columna con más faltantes y averigua **si se reparten al azar o se concentran en algún grupo**. Usa `pd.crosstab` contra alguna columna categórica, como en el Ejercicio 1 del cuaderno resuelto.

Es la diferencia entre poder imputar y tener que declarar.

In [4]:
# Tu código aquí
# Cruce para "Comuna Social"
pd.crosstab(df['Mes'], df['Comuna Social'].isna(), normalize= 'index')

Comuna Social,False,True
Mes,,
Abril,1.000000,0.000000
Agosto,1.000000,0.000000
Diciembre,1.000000,0.000000
Enero,1.000000,0.000000
Febrero,1.000000,0.000000
Julio,0.999948,0.000052
Junio,1.000000,0.000000
Marzo,0.999940,0.000060
Mayo,1.000000,0.000000


In [5]:
# Cruce para "Region Social"
pd.crosstab(df['Mes'], df['Region Social'].isna(), normalize= 'index')

Region Social,False,True
Mes,,
Abril,1.000000,0.000000
Agosto,1.000000,0.000000
Diciembre,1.000000,0.000000
Enero,1.000000,0.000000
Febrero,1.000000,0.000000
Julio,0.999948,0.000052
Junio,1.000000,0.000000
Marzo,0.999940,0.000060
Mayo,1.000000,0.000000


### 1.2 Exactitud — rangos y valores imposibles

In [6]:
df.describe().T.round(2)

,count,mean,min,25%,50%,75%,max,std
ID,202406.0,6005702.478928,4479479.0,5794956.25,6005425.5,6215138.75,6438180.0,243160.81305
Fecha de actuacion (1era firma),202406,2025-07-05 07:52:49.435688192,2023-05-24 00:00:00,2025-04-07 00:00:00,2025-07-10 00:00:00,2025-10-03 00:00:00,2025-12-31 00:00:00,NaN
Fecha de registro (ultima firma),202406,2025-07-05 13:26:14.028437760,2023-05-24 00:00:00,2025-04-08 00:00:00,2025-07-10 00:00:00,2025-10-03 00:00:00,2025-12-31 00:00:00,NaN
Fecha de aprobacion x SII,202406,2025-07-05 14:46:33.759078400,2025-01-01 00:00:00,2025-04-08 00:00:00,2025-07-10 00:00:00,2025-10-03 00:00:00,2025-12-31 00:00:00,NaN
Anio,202406.0,2025.0,2025.0,2025.0,2025.0,2025.0,2025.0,0.0
Region Tributaria,202406.0,9.794156,1.0,6.0,12.0,13.0,16.0,3.896464
Capital,202406.0,6331926.902236,1.0,1000000.0,1500000.0,5000000.0,50000000000.0,127586661.624422
Region Social,202403.0,9.792459,1.0,6.0,12.0,13.0,16.0,3.896971


### ✏️ Actividad 1.2

Mira los mínimos y máximos. **¿Hay valores imposibles para lo que la columna representa?** Busca en particular valores centinela: `-1`, `0`, `999`, `9999`, `-999`, o cadenas como `'S/I'`, `'NULL'`, `'sin dato'`.

Recuerda: un centinela numérico **no** se delata en los tipos de dato.

In [7]:
# Tu código aquí. Sugerencia:
for col in df.select_dtypes(include='number').columns:
    print(col, sorted(df[col].dropna().unique())[:5], '...', sorted(df[col].dropna().unique())[-5:])


ID [np.int64(4479479), np.int64(4833738), np.int64(5114947), np.int64(5134403), np.int64(5138749)] ... [np.int64(6438165), np.int64(6438167), np.int64(6438173), np.int64(6438179), np.int64(6438180)]
Anio [np.int64(2025)] ... [np.int64(2025)]
Region Tributaria [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)] ... [np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16)]
Capital [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)] ... [np.int64(5000000000), np.int64(6000000000), np.int64(8000000000), np.int64(15000000000), np.int64(50000000000)]
Region Social [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)] ... [np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(16.0)]


### 1.3 Consistencia — reglas cruzadas

### ✏️ Actividad 1.3

Escribe **dos reglas de consistencia** que tu conjunto debería cumplir y verifícalas. Ejemplos:

- «La fecha de término no puede ser anterior a la de inicio.»
- «Si el estado es *cerrado*, el monto vigente debe estar vacío.»
- «La suma de las partes debe igualar el total.»

Estas reglas no salen de los datos: salen de entender qué representan.

In [8]:
# Regla 1: Fecha de registro debe ser mayor o igual a Fecha de actuacion
compararFechas = df.loc[df['Fecha de registro (ultima firma)'] < df['Fecha de actuacion (1era firma)'],
                        ['Razon Social', 'Comuna Social', 'Region Social', 'Capital',
                         'Fecha de actuacion (1era firma)', 'Fecha de registro (ultima firma)']]

print('Nro. de fechas de registro (última firma) menores a su fecha de actuación (primera firma): ',
      len(compararFechas))

compararFechas.head()

print('\n-------------------------------------------------------------------------------------------')

# Regla 2: Fecha de aprobación SII debe ser mayor o igual a Fecha de registro
anioFechas = df.loc[df['Fecha de aprobacion x SII'] < df['Fecha de registro (ultima firma)'],
                    ['Razon Social', 'Comuna Social', 'Region Social', 'Capital',
                     'Fecha de registro (ultima firma)', 'Fecha de aprobacion x SII']]

print('\nNro. de fechas de registro menores al año 2025: ',
      len(anioFechas))

anioFechas.head(15)

Nro. de fechas de registro (última firma) menores a su fecha de actuación (primera firma):  0

-------------------------------------------------------------------------------------------

Nro. de fechas de registro menores al año 2025:  15


,Razon Social,Comuna Social,Region Social,Capital,Fecha de registro (ultima firma),Fecha de aprobacion x SII
92500,SERVICIOS GARFU SPA,INDEPENDENCIA,13.0,10000000,2025-06-26,2025-06-25
92649,Dayanna Escalona SpA,CONCEPCIÓN,8.0,5000000,2025-06-27,2025-06-26
93607,PERZA SpA,MAULE,7.0,1000000,2025-06-28,2025-06-27
95351,Ecoconstrucciones La Manada SpA,MAIPU,13.0,900000,2025-07-02,2025-07-01
98945,Millagua Spa,LO ESPEJO,13.0,1000000,2025-07-09,2025-07-08
100626,Maria Adriana Navarro González e hijos spa,BUIN,13.0,500000,2025-07-11,2025-07-10
101462,INVERSIONES JPMG SpA,MAIPU,13.0,500000,2025-07-12,2025-07-11
103238,ASESORES CONNECT SPA,PEÑALOLEN,13.0,15000000,2025-07-16,2025-07-15
103830,DISEÑO Y FABRICACION MARCELO ISAIAS SILVA DIAZ...,VALDIVIA,14.0,1100000,2025-07-17,2025-07-16
103989,André Paul SpA,CHILLAN,16.0,2000000,2025-07-17,2025-07-16


### 1.4 Unicidad

In [9]:
print('Duplicados exactos: ', df.duplicated().sum())
# Si tu conjunto tiene una columna identificadora, revísala también:
print('ID repetidos: ', df['ID'].duplicated().sum())
print('RUTs repetidos: ', df['RUT'].duplicated().sum())

Duplicados exactos:  0
ID repetidos:  0
RUTs repetidos:  0


### ✏️ Actividad 1.4

Construye una **clave normalizada** (sin espacios, en mayúsculas, sin tildes) con las columnas que deberían identificar un registro, y cuenta los casi-duplicados.

In [10]:
def normalizar_texto(serie):
    return (serie.astype(str).str.strip().str.upper()
                 .str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii'))

# Tu código aquí
clave = normalizar_texto(df['RUT']) + '|' + normalizar_texto(df['Razon Social'])
df = df.loc[~clave.duplicated()].copy()

print('Registros repetidos tras normalizar la clave:', clave.duplicated().sum())

Registros repetidos tras normalizar la clave: 0


### 1.5 Oportunidad

In [11]:
print('Si tienes alguna columna de fecha, ¿cuál es el registro más reciente?')
print('\nFecha de registro más reciente: ', df['Fecha de registro (ultima firma)'].max())
print('\n------------------------------------------------------------------------')
print('\n¿El conjunto está vigente para la pregunta que quieres responder?')
print('\nSí, está dentro del rango del año analizado (2025).')


Si tienes alguna columna de fecha, ¿cuál es el registro más reciente?

Fecha de registro más reciente:  2025-12-31 00:00:00

------------------------------------------------------------------------

¿El conjunto está vigente para la pregunta que quieres responder?

Sí, está dentro del rango del año analizado (2025).


---
## Parte 2 · Limpieza, decisión por decisión

Ahora corriges. **Regla del laboratorio: cada celda de esta parte tiene su fila en la bitácora.** Si aplicas una transformación que no registraste, no cuenta.

Recuerda las tres decisiones legítimas: **Corregir**, **Descartar** o **Declarar como limitación**.

### ✏️ Actividad 2.1 — Centinelas a faltantes

In [12]:
# Tu código aquí

# Reemplazamos faltantes por NaN en "Comuna Social"
df['Comuna Social'] = df['Comuna Social'].replace('', np.nan)
df['Comuna Social'] = df['Comuna Social'].replace(' ', np.nan)

# Reemplazamos faltantes por NaN en "Region Social"
df['Region Social'] = df['Region Social'].replace('', np.nan)
df['Region Social'] = df['Region Social'].replace(' ', np.nan)

print('Faltantes tras la conversión:')
print(df[['Comuna Social', 'Region Social']].isna().sum())

Faltantes tras la conversión:
Comuna Social    3
Region Social    3
dtype: int64


In [13]:
# Y eliminamos las filas con datos faltantes
cantidadFaltantes = len(df)
df = df.dropna(subset= ['Comuna Social', 'Region Social'])
print(f'Filas con faltantes eliminadas: {cantidadFaltantes - len(df)}')

Filas con faltantes eliminadas: 3


In [14]:
# Verificamos el nuevo porcentaje de faltantes tras la limpieza

faltantes = df.isna().sum().to_frame('faltantes')
faltantes['pct'] = (faltantes['faltantes'] / len(df) * 100).round(2)
faltantes.sort_values('faltantes', ascending=False)

,faltantes,pct
ID,0,0.0
RUT,0,0.0
Razon Social,0,0.0
Fecha de actuacion (1era firma),0,0.0
Fecha de registro (ultima firma),0,0.0
Fecha de aprobacion x SII,0,0.0
Anio,0,0.0
Mes,0,0.0
Comuna Tributaria,0,0.0
Region Tributaria,0,0.0


In [15]:
# Observamos la nueva forma de la planilla de datos
print('Nueva forma del Dataset tras eliminar datos faltantes: ', df.shape)
df.dtypes

Nueva forma del Dataset tras eliminar datos faltantes:  (202403, 15)


,0
ID,int64
RUT,object
Razon Social,object
Fecha de actuacion (1era firma),datetime64[ns]
Fecha de registro (ultima firma),datetime64[ns]
Fecha de aprobacion x SII,datetime64[ns]
Anio,int64
Mes,object
Comuna Tributaria,object
Region Tributaria,int64


### ✏️ Actividad 2.2 — Normalización de categorías

Normaliza mecánicamente y después revisa lo que queda: los plurales, las abreviaturas y los sinónimos necesitan un diccionario de equivalencias que escribes tú.

In [16]:
# Tu código aquí

# Valores distintos en columna Region Social (deberían ser 16)
print('Valores distintos declarados en "Región Social": ', df['Region Social'].nunique())
sorted(df['Region Social'].unique())

# Valores distintos en columna Region Social (deberían ser 16)
print('\nValores distintos declarados en "Región Tributaria": ', df['Region Tributaria'].nunique())
sorted(df['Region Tributaria'].unique())

# Valores distintos en columna Comuna Social (deberían ser 346)
print('\nValores distintos declarados en "Comuna Social": ', df['Comuna Social'].nunique())
sorted(df['Comuna Social'].unique())

# Valores distintos en columna Comuna Tributaria (deberían ser 346)
print('\nValores distintos declarados en "Comuna Tributaria": ', df['Comuna Tributaria'].nunique())

# Ver todos los valores de la columna "Comuna Social"
print('\nTodas las 346 comunas: ')
sorted(df['Comuna Social'].unique())

Valores distintos declarados en "Región Social":  16

Valores distintos declarados en "Región Tributaria":  16

Valores distintos declarados en "Comuna Social":  346

Valores distintos declarados en "Comuna Tributaria":  346

Todas las 346 comunas: 


['ALGARROBO',
 'ALHUE',
 'ALTO BIO BIO',
 'ALTO DEL CARMEN',
 'ALTO HOSPICIO',
 'ANCUD',
 'ANDACOLLO',
 'ANGOL',
 'ANTARTIDA',
 'ANTOFAGASTA',
 'ANTUCO',
 'ARAUCO',
 'ARICA',
 'AYSEN',
 'BUIN',
 'BULNES',
 'CABILDO',
 'CABO DE HORNOS',
 'CABRERO',
 'CALAMA',
 'CALBUCO',
 'CALDERA',
 'CALERA DE TANGO',
 'CALLE LARGA',
 'CAMARONES',
 'CAMINA',
 'CANELA',
 'CANETE',
 'CARAHUE',
 'CARTAGENA',
 'CASABLANCA',
 'CASTRO',
 'CATEMU',
 'CAUQUENES',
 'CERRILLOS',
 'CERRO NAVIA',
 'CHAITEN',
 'CHANCO',
 'CHAÑARAL',
 'CHEPICA',
 'CHIGUAYANTE',
 'CHILE CHICO',
 'CHILLAN',
 'CHILLAN VIEJO',
 'CHIMBARONGO',
 'CHOLCHOL',
 'CHONCHI',
 'CISNES',
 'COBQUECURA',
 'COCHAMO',
 'COCHRANE',
 'CODEGUA',
 'COELEMU',
 'COIHUECO',
 'COINCO',
 'COLBUN',
 'COLCHANE',
 'COLINA',
 'COLLIPULLI',
 'COLTAUCO',
 'COMBARBALA',
 'CON CON',
 'CONCEPCIÓN',
 'CONCHALI',
 'CONSTITUCION',
 'CONTULMO',
 'COPIAPO',
 'COQUIMBO',
 'CORONEL',
 'CORRAL',
 'COYHAIQUE',
 'CUNCO',
 'CURACAUTIN',
 'CURACAVI',
 'CURACO DE VELEZ',
 'CURANIL

In [17]:
# Normalizamos Comuna Social y Comuna Tributaria
def normalizar_comuna(serie):
    """Quita espacios, unifica mayúsculas y elimina tildes."""
    return (serie.astype(str)
                 .str.strip()
                 .str.upper()
                 .str.normalize('NFKD')
                 .str.encode('ascii', 'ignore')
                 .str.decode('ascii'))

df['comuna_social_norm'] = normalizar_comuna(df['Comuna Social'])
df['comuna_tributaria_norm']  = normalizar_comuna(df['Comuna Tributaria'])

print('Comuna Social:', df['Comuna Social'].nunique(), '->', df['comuna_social_norm'].nunique())
print('Comuna Tributaria:', df['Comuna Tributaria'].nunique(),  '->', df['comuna_tributaria_norm'].nunique())

Comuna Social: 346 -> 346
Comuna Tributaria: 346 -> 346


In [18]:
print('Todas las 346 comunas normalizadas: ')
sorted(df['comuna_social_norm'].unique())

Todas las 346 comunas normalizadas: 


['ALGARROBO',
 'ALHUE',
 'ALTO BIO BIO',
 'ALTO DEL CARMEN',
 'ALTO HOSPICIO',
 'ANCUD',
 'ANDACOLLO',
 'ANGOL',
 'ANTARTIDA',
 'ANTOFAGASTA',
 'ANTUCO',
 'ARAUCO',
 'ARICA',
 'AYSEN',
 'BUIN',
 'BULNES',
 'CABILDO',
 'CABO DE HORNOS',
 'CABRERO',
 'CALAMA',
 'CALBUCO',
 'CALDERA',
 'CALERA DE TANGO',
 'CALLE LARGA',
 'CAMARONES',
 'CAMINA',
 'CANELA',
 'CANETE',
 'CARAHUE',
 'CARTAGENA',
 'CASABLANCA',
 'CASTRO',
 'CATEMU',
 'CAUQUENES',
 'CERRILLOS',
 'CERRO NAVIA',
 'CHAITEN',
 'CHANARAL',
 'CHANCO',
 'CHEPICA',
 'CHIGUAYANTE',
 'CHILE CHICO',
 'CHILLAN',
 'CHILLAN VIEJO',
 'CHIMBARONGO',
 'CHOLCHOL',
 'CHONCHI',
 'CISNES',
 'COBQUECURA',
 'COCHAMO',
 'COCHRANE',
 'CODEGUA',
 'COELEMU',
 'COIHUECO',
 'COINCO',
 'COLBUN',
 'COLCHANE',
 'COLINA',
 'COLLIPULLI',
 'COLTAUCO',
 'COMBARBALA',
 'CON CON',
 'CONCEPCION',
 'CONCHALI',
 'CONSTITUCION',
 'CONTULMO',
 'COPIAPO',
 'COQUIMBO',
 'CORONEL',
 'CORRAL',
 'COYHAIQUE',
 'CUNCO',
 'CURACAUTIN',
 'CURACAVI',
 'CURACO DE VELEZ',
 'CURANIL

### ✏️ Actividad 2.3 — Duplicados

In [20]:
# Tu código aquí

# Para corregir datos duplicados use este código
conDuplicados = len(df)
df = df.loc[~clave.duplicated()].copy()
print(f'Nro. filas: {conDuplicados} -> {len(df)}  ({conDuplicados - len(df)} eliminadas)')

Nro. filas: 202403 -> 202403  (0 eliminadas)


### ✏️ Actividad 2.4 — Tipos de dato

Convierte a numérico lo que deba ser numérico y a fecha lo que deba ser fecha. Si hay formatos mixtos, prueba formatos explícitos en orden en vez de dejar que pandas adivine.

In [21]:
# Tu código aquí

# Revisamos las regiones de "Region Social"
print('Regiones de "Region Social": ')
sorted(df['Region Social'].unique())

Regiones de "Region Social": 


[np.float64(1.0),
 np.float64(2.0),
 np.float64(3.0),
 np.float64(4.0),
 np.float64(5.0),
 np.float64(6.0),
 np.float64(7.0),
 np.float64(8.0),
 np.float64(9.0),
 np.float64(10.0),
 np.float64(11.0),
 np.float64(12.0),
 np.float64(13.0),
 np.float64(14.0),
 np.float64(15.0),
 np.float64(16.0)]

In [22]:
# Las regiones tienen un decimal innecesario, se corrige
df['Region Social'] = df['Region Social'].astype(int)


In [23]:
# Verificamos la conversión
df.dtypes

,0
ID,int64
RUT,object
Razon Social,object
Fecha de actuacion (1era firma),datetime64[ns]
Fecha de registro (ultima firma),datetime64[ns]
Fecha de aprobacion x SII,datetime64[ns]
Anio,int64
Mes,object
Comuna Tributaria,object
Region Tributaria,int64


### ✏️ Actividad 2.5 — Atípicos

Identifica los valores extremos y **pronúnciate sobre cada grupo**: ¿error de registro o fenómeno real?

Cruza la variable con otra que te permita juzgar plausibilidad, como en el Ejercicio 8. No elimines atípicos solo por serlo.

In [30]:
# Tu código aquí

# Observamos el límite inferior de la columna "Capital" (bajo el 25% de los datos)
q1, q3 = df['Capital'].quantile([0.25, 0.75])
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
df.loc[df['Capital'] < limite_inferior]

,ID,RUT,Razon Social,Fecha de actuacion (1era firma),Fecha de registro (ultima firma),Fecha de aprobacion x SII,Anio,Mes,Comuna Tributaria,Region Tributaria,Codigo de sociedad,Tipo de actuacion,Capital,Comuna Social,Region Social,comuna_social_norm,comuna_tributaria_norm


In [27]:
# Observamos el límite superior de la columna "Capital" (sobre el 75% de los datos)
q1, q3 = df['Capital'].quantile([0.25, 0.75])
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr
df.loc[df['Capital'] > limite_superior]

,ID,RUT,Razon Social,Fecha de actuacion (1era firma),Fecha de registro (ultima firma),Fecha de aprobacion x SII,Anio,Mes,Comuna Tributaria,Region Tributaria,Codigo de sociedad,Tipo de actuacion,Capital,Comuna Social,Region Social,comuna_social_norm,comuna_tributaria_norm
15,5597977,78081214-1,MONTAJES LOS VOLCANES SpA,2025-01-01,2025-01-01,2025-01-01,2025,Enero,ÑUÑOA,13,SpA,CONSTITUCIÓN,75000000,ÑUÑOA,13,NUNOA,NUNOA
18,5597743,78081175-7,INVERSIONES PUNILLA SpA,2025-01-01,2025-01-01,2025-01-01,2025,Enero,CONCEPCIÓN,8,SpA,CONSTITUCIÓN,100000000,CONCEPCIÓN,8,CONCEPCION,CONCEPCION
19,5598081,78081245-1,"Servicio de alineación, balanceo y aire acondi...",2025-01-01,2025-01-01,2025-01-01,2025,Enero,COQUIMBO,4,EIRL,CONSTITUCIÓN,40000000,COQUIMBO,4,COQUIMBO,COQUIMBO
32,5596069,78081199-4,INVERSIONES PRADO Y ROMÁN LIMITADA,2024-12-30,2025-01-01,2025-01-01,2025,Enero,LAS CONDES,13,SRL,CONSTITUCIÓN,80000000,LAS CONDES,13,LAS CONDES,LAS CONDES
50,5598054,78081243-5,Transportes NIcolás Rafael Abarca Pino E.I.R.L.,2025-01-01,2025-01-01,2025-01-01,2025,Enero,PUENTE ALTO,13,EIRL,CONSTITUCIÓN,15000000,PUENTE ALTO,13,PUENTE ALTO,PUENTE ALTO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202339,6437081,78325082-9,Holding Inversiones Integrales SpA,2025-12-31,2025-12-31,2025-12-31,2025,Diciembre,PUERTO MONTT,10,SpA,CONSTITUCIÓN,35000000,PUERTO MONTT,10,PUERTO MONTT,PUERTO MONTT
202366,6420204,78325116-7,Constructora Yazca Limitada,2025-12-30,2025-12-31,2025-12-31,2025,Diciembre,TEMUCO,9,SRL,CONSTITUCIÓN,32000000,TEMUCO,9,TEMUCO,TEMUCO
202378,6436858,78325182-5,NOVASEC INGENIERIA SPA,2025-12-31,2025-12-31,2025-12-31,2025,Diciembre,CORONEL,8,SpA,CONSTITUCIÓN,15000000,CORONEL,8,CORONEL,CORONEL
202386,6436875,78325326-7,S&F NEXOS EDUCACIÓN SpA,2025-12-31,2025-12-31,2025-12-31,2025,Diciembre,VALPARAISO,5,SpA,CONSTITUCIÓN,15000000,VALPARAISO,5,VALPARAISO,VALPARAISO


_Tu conclusión sobre los atípicos:_ En conclusión, lo único que podría llamar la atención en el DataFrame, son los valores de la columna "Capital", pero si nos vamos a lo legal, no hay un mínimo fijado para el capital de una empresa, solo hay excepciones para las sociedades anónimas especiales, como por ejemplo los bancos.



---
## Parte 3 · Transformación y nuevos atributos

Preparar no es solo corregir: también es **construir** lo que el análisis va a necesitar.

### ✏️ Actividad 3.1 — Un atributo derivado

Crea **al menos una variable nueva** a partir de las existentes, que aporte a tu pregunta analítica. Ejemplos: una razón entre dos columnas, la antigüedad en años a partir de una fecha, un tramo a partir de un valor continuo.

Explica en la celda siguiente por qué esa variable ayuda a responder tu pregunta.

In [31]:
# Tu código aquí

# Nuevo atributo: dias_tramitacion_registro
df['dias_tramitacion_registro'] = (
    df['Fecha de aprobacion x SII'] - df['Fecha de actuacion (1era firma)']
).dt.days

In [32]:
df.dtypes

,0
ID,int64
RUT,object
Razon Social,object
Fecha de actuacion (1era firma),datetime64[ns]
Fecha de registro (ultima firma),datetime64[ns]
Fecha de aprobacion x SII,datetime64[ns]
Anio,int64
Mes,object
Comuna Tributaria,object
Region Tributaria,int64


_Por qué esta variable aporta a mi pregunta:_ La variable **dias_tramitacion_registro** permitirá obtener el promedio por comuna entre los días en que una empresa nace en su concepción (primera firma) hasta que se registra legalmente en el sistema tributario.



### ✏️ Actividad 3.2 — Normalización o codificación

Si tu conjunto tiene variables numéricas en escalas muy distintas, o categóricas que necesitarás codificar en la semana 4, déjalas preparadas aquí.

In [ ]:
# Tu código aquí. Referencia:
# from sklearn.preprocessing import StandardScaler, MinMaxScaler
# df_cod = pd.get_dummies(df, columns=['TU_COLUMNA_CATEGORICA'], drop_first=True)


---
## Parte 4 · El flujo como función reutilizable

Junta todo lo anterior en **una sola función**. Esto es lo que evalúa el RA3 y lo que se revisa en la Sección A de la Entrega Final: que tu limpieza se pueda volver a ejecutar de principio a fin.

In [33]:
def limpiar(ruta, sep=SEPARADOR, semilla=SEMILLA):
    """Flujo completo de limpieza de mi conjunto. Devuelve (df_limpio, informe)."""
    np.random.seed(semilla)
    d = pd.read_excel(ruta)
    informe = {'filas_iniciales': len(d)}

    # 1 - Descartar datos faltantes
    # Reemplazamos faltantes por NaN en "Comuna Social"
    d['Comuna Social'] = d['Comuna Social'].replace('', np.nan)
    d['Comuna Social'] = d['Comuna Social'].replace(' ', np.nan)

    # Reemplazamos faltantes por NaN en "Region Social"
    d['Region Social'] = d['Region Social'].replace('', np.nan)
    d['Region Social'] = d['Region Social'].replace(' ', np.nan)

    # Y eliminamos las filas con datos faltantes
    cantidadFaltantes = len(d)
    d = d.dropna(subset= ['Comuna Social', 'Region Social'])


    # 2 - Normalizamos las comunas eliminando tildes y espacios sobrantes
    d['comuna_social_norm'] = normalizar_comuna(d['Comuna Social'])
    d['comuna_tributaria_norm']  = normalizar_comuna(d['Comuna Tributaria'])


    # 3 Para corregir datos duplicados
    conDuplicados = len(d)
    d = d.loc[~clave.duplicated()].copy()
    print(f'Nro. filas: {conDuplicados} -> {len(d)}  ({conDuplicados - len(d)} eliminadas)')


    # 4 - Tipos de datos
    # Las regiones tienen un decimal innecesario, se corrige
    d['Region Social'] = d['Region Social'].astype(int)


    # 5 · atípicos
    # No aplica, ya que, en el diagnóstico no se encontraron valores atípicos


    # 6 - Atributos derivados
    # Nuevo atributo: dias_tramitacion_registro
    d['dias_tramitacion_registro'] = (
      d['Fecha de aprobacion x SII'] - d['Fecha de actuacion (1era firma)']
    ).dt.days

    informe['filas_finales'] = len(d)
    return d, informe


limpio, informe = limpiar(ARCHIVO)
print(informe)

Nro. filas: 202403 -> 202403  (0 eliminadas)
{'filas_iniciales': 202406, 'filas_finales': 202403}


In [34]:
# Comprobación: ejecuta la función DOS VECES y verifica que da lo mismo.
a, _ = limpiar(ARCHIVO)
b, _ = limpiar(ARCHIVO)
print('¿Reproducible?', a.equals(b))

Nro. filas: 202403 -> 202403  (0 eliminadas)
Nro. filas: 202403 -> 202403  (0 eliminadas)
¿Reproducible? True


In [36]:
# Guarda el conjunto limpio. Este archivo es el insumo de la semana 3.
limpio.to_excel('mi_conjunto_limpio.xlsx', index=False)
print('Guardado:', limpio.shape)

Guardado: (202403, 18)


---
## Parte 5 · Versionado

Sube el cuaderno y el conjunto limpio a tu repositorio. Un commit con mensaje descriptivo, no «cambios».

In [ ]:
# En una terminal:
# git add S2_Notebook_Laboratorio_CDD.ipynb mi_conjunto_limpio.csv
# git commit -m 'Semana 2: diagnostico de calidad y flujo de limpieza'
# git push


**Enlace a mi repositorio:** _https://github.com/beneducarrco99/Laboratorio_2_Semana_2_Ciencia_de_Datos_Carrasco_Benjamin_

---

## Antes de cerrar el laboratorio

- [ ] Las cinco dimensiones diagnosticadas y registradas en la hoja «Resumen de calidad»
- [ ] Cada transformación aplicada tiene su fila en la bitácora, con justificación
- [ ] Me pronuncié sobre los atípicos: cuáles son error y cuáles fenómeno real
- [ ] Creé al menos un atributo derivado y expliqué para qué sirve
- [ ] La función `limpiar()` corre de principio a fin y entrega lo mismo dos veces
- [ ] El conjunto limpio está guardado y versionado
- [ ] Tengo anotadas las limitaciones que declararé en el informe

Lo que quede pendiente, al Foro de Consultas o a la sesión sincrónica.

_Ciencia de Datos · Advance UCSC · Semana 2_